# Claude API 툴콜링 (Tool Use)

Claude가 우리가 정의한 함수(툴)를 호출하며 작업을 수행하는 예제입니다.

**시나리오**: 온라인 서점 주문 도우미
- `search_book` — 책 제목으로 검색
- `check_stock` — 책 ID로 재고 확인
- `place_order` — 주문 실행

"책 찾아서 재고 있으면 주문해줘"라는 요청을 처리하려면 **검색 → 재고 확인 → 주문** 순으로
툴을 3번 연쇄 호출해야 합니다. 각 단계의 결과가 다음 단계의 입력이 되기 때문입니다.

## 1. 클라이언트 초기화

In [2]:
import json
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), ".env 파일에 ANTHROPIC_API_KEY를 설정해주세요"

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"
print("클라이언트 준비 완료")

클라이언트 준비 완료


## 2. 툴 구현 (가짜 서점 백엔드)

실제 서비스라면 DB나 외부 API를 호출하겠지만, 여기서는 인메모리 데이터로 흉내냅니다.

In [3]:
BOOKS_DB = {
    "BK-1023": {"title": "파이썬 클린코드", "author": "마리아노 아나야", "price": 28000, "stock": 7},
    "BK-2041": {"title": "러닝 타입스크립트", "author": "조시 골드버그", "price": 32000, "stock": 0},
    "BK-3310": {"title": "데이터 중심 애플리케이션 설계", "author": "마틴 클레프만", "price": 48000, "stock": 3},
}

ORDERS = []


def search_book(title: str) -> str:
    """제목에 검색어가 포함된 책을 찾아 반환"""
    for book_id, info in BOOKS_DB.items():
        if title.strip() in info["title"]:
            return json.dumps(
                {"book_id": book_id, "title": info["title"], "author": info["author"], "price": info["price"]},
                ensure_ascii=False,
            )
    return json.dumps({"error": f"'{title}' 검색 결과가 없습니다"}, ensure_ascii=False)


def check_stock(book_id: str) -> str:
    """책 ID로 현재 재고 수량 확인"""
    if book_id not in BOOKS_DB:
        return json.dumps({"error": f"존재하지 않는 book_id: {book_id}"}, ensure_ascii=False)
    return json.dumps({"book_id": book_id, "stock": BOOKS_DB[book_id]["stock"]}, ensure_ascii=False)


def place_order(book_id: str, quantity: int) -> str:
    """주문을 생성하고 주문 정보를 반환"""
    if book_id not in BOOKS_DB:
        return json.dumps({"error": f"존재하지 않는 book_id: {book_id}"}, ensure_ascii=False)
    book = BOOKS_DB[book_id]
    if book["stock"] < quantity:
        return json.dumps({"error": f"재고 부족 (현재 {book['stock']}권)"}, ensure_ascii=False)
    book["stock"] -= quantity
    order = {
        "order_id": f"ORD-{len(ORDERS) + 1:04d}",
        "book_id": book_id,
        "title": book["title"],
        "quantity": quantity,
        "total": book["price"] * quantity,
        "status": "confirmed",
    }
    ORDERS.append(order)
    return json.dumps(order, ensure_ascii=False)


TOOL_FUNCTIONS = {"search_book": search_book, "check_stock": check_stock, "place_order": place_order}
print("툴 구현 완료:", list(TOOL_FUNCTIONS))

툴 구현 완료: ['search_book', 'check_stock', 'place_order']


## 3. 툴 스키마 정의

Claude에게 전달할 툴 명세입니다. `description`이 상세할수록 Claude가 언제/어떻게 쓸지 잘 판단합니다.

In [4]:
tools = [
    {
        "name": "search_book",
        "description": "제목으로 책을 검색합니다. 사용자가 책을 언급하면 가장 먼저 이 툴로 book_id를 알아내세요.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "검색할 책 제목 (부분 일치 가능)"}
            },
            "required": ["title"],
        },
    },
    {
        "name": "check_stock",
        "description": "book_id로 현재 재고 수량을 확인합니다. 주문하기 전에 반드시 재고를 확인하세요.",
        "input_schema": {
            "type": "object",
            "properties": {
                "book_id": {"type": "string", "description": "search_book으로 얻은 책 ID (예: BK-1023)"}
            },
            "required": ["book_id"],
        },
    },
    {
        "name": "place_order",
        "description": "책을 주문합니다. 재고 확인 후 충분한 경우에만 호출하세요.",
        "input_schema": {
            "type": "object",
            "properties": {
                "book_id": {"type": "string", "description": "주문할 책 ID"},
                "quantity": {"type": "integer", "description": "주문 수량"},
            },
            "required": ["book_id", "quantity"],
        },
    },
]
print(f"툴 {len(tools)}개 정의 완료")

툴 3개 정의 완료


## 4. 에이전틱 루프 실행

Claude가 `stop_reason == "tool_use"`로 응답하면 → 툴 실행 → 결과를 돌려주고 → 다시 요청.
Claude가 더 이상 툴을 부르지 않을 때까지 반복합니다.

아래 요청은 **검색(1) → 재고 확인(2) → 주문(3)** 총 3번의 툴 호출로 이어집니다.

In [5]:
user_request = "'파이썬 클린코드'라는 책을 찾아서, 재고가 있으면 2권 주문해줘."

messages = [{"role": "user", "content": user_request}]
tool_call_count = 0
MAX_ROUNDS = 10  # 무한 루프 방지 안전장치

for round_num in range(1, MAX_ROUNDS + 1):
    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        tools=tools,
        messages=messages,
    )

    # 툴 호출이 없으면 최종 응답이므로 종료
    if response.stop_reason != "tool_use":
        break

    # 어시스턴트 턴(tool_use 블록 포함)을 이력에 추가
    messages.append({"role": "assistant", "content": response.content})

    # 이번 턴의 모든 tool_use 블록을 실행하고, 결과는 하나의 user 메시지로 묶어 반환
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            tool_call_count += 1
            print(f"[툴 호출 {tool_call_count}] {block.name}({json.dumps(block.input, ensure_ascii=False)})")
            result = TOOL_FUNCTIONS[block.name](**block.input)
            print(f"  → 결과: {result}\n")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result,
            })

    messages.append({"role": "user", "content": tool_results})

print("=" * 50)
print(f"총 툴 호출 횟수: {tool_call_count}")
print("=" * 50)
print("\n[최종 응답]")
for block in response.content:
    if block.type == "text":
        print(block.text)

[툴 호출 1] search_book({"title": "파이썬 클린코드"})
  → 결과: {"book_id": "BK-1023", "title": "파이썬 클린코드", "author": "마리아노 아나야", "price": 28000}

[툴 호출 2] check_stock({"book_id": "BK-1023"})
  → 결과: {"book_id": "BK-1023", "stock": 7}

[툴 호출 3] place_order({"book_id": "BK-1023", "quantity": 2})
  → 결과: {"order_id": "ORD-0001", "book_id": "BK-1023", "title": "파이썬 클린코드", "quantity": 2, "total": 56000, "status": "confirmed"}

총 툴 호출 횟수: 3

[최종 응답]
주문이 완료되었습니다! 📚

**주문 내역**
- 책: 파이썬 클린코드 (마리아노 아나야)
- 수량: 2권
- 권당 가격: 28,000원
- **총 결제 금액: 56,000원**
- 주문번호: ORD-0001
- 상태: 주문 확정 ✅

재고가 7권 있어 문제없이 2권 주문되었습니다. 더 필요하신 게 있으면 말씀해주세요!


## 5. 에러 케이스: 재고가 없는 책

재고가 0인 책을 주문하면 Claude가 재고 확인 결과를 보고 주문을 진행하지 않는지 확인해봅니다.

In [ ]:
messages = [{"role": "user", "content": "'러닝 타입스크립트' 1권 주문해줘."}]

for _ in range(MAX_ROUNDS):
    response = client.messages.create(model=MODEL, max_tokens=16000, tools=tools, messages=messages)
    if response.stop_reason != "tool_use":
        break
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = TOOL_FUNCTIONS[block.name](**block.input)
            print(f"[툴 호출] {block.name}({json.dumps(block.input, ensure_ascii=False)}) → {result}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
    messages.append({"role": "user", "content": tool_results})

print("\n[최종 응답]")
for block in response.content:
    if block.type == "text":
        print(block.text)

## 6. (보너스) Tool Runner — 루프를 SDK에 맡기기

`@beta_tool` 데코레이터를 쓰면 함수 시그니처와 docstring에서 스키마가 자동 생성되고,
`tool_runner`가 위에서 직접 짰던 에이전틱 루프를 대신 처리해줍니다.

In [ ]:
from anthropic import beta_tool


@beta_tool
def search_book_rt(title: str) -> str:
    """제목으로 책을 검색해 book_id, 가격 등을 반환합니다.

    Args:
        title: 검색할 책 제목 (부분 일치 가능)
    """
    return search_book(title)


@beta_tool
def check_stock_rt(book_id: str) -> str:
    """book_id로 현재 재고 수량을 확인합니다. 주문 전에 반드시 호출하세요.

    Args:
        book_id: search_book_rt로 얻은 책 ID
    """
    return check_stock(book_id)


@beta_tool
def place_order_rt(book_id: str, quantity: int) -> str:
    """책을 주문합니다. 재고가 충분한 경우에만 호출하세요.

    Args:
        book_id: 주문할 책 ID
        quantity: 주문 수량
    """
    return place_order(book_id, quantity)


runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=16000,
    tools=[search_book_rt, check_stock_rt, place_order_rt],
    messages=[{"role": "user", "content": "'데이터 중심 애플리케이션 설계' 재고 확인하고 1권 주문해줘."}],
)

# 루프가 자동으로 돌고, 각 반복의 메시지를 순회할 수 있습니다
for message in runner:
    for block in message.content:
        if block.type == "tool_use":
            print(f"[툴 호출] {block.name}({json.dumps(block.input, ensure_ascii=False)})")
        elif block.type == "text":
            print(f"[응답] {block.text}")